# فاز ۴: مراقبت‌های پیشگیرانه و پایش دینامیک التیام طولی زخم
### Phase 4: AI-Assisted Preventive Care, Longitudinal Healing Trajectory & CDSS

این نوت‌بوک پاسخ مستقیم به بخش **«مراقبت‌های پیشگیرانه»** در عنوان پژوهشی است. در این ماژول، تغییرات زمانی بافت‌ها در طول ویزیت‌های متوالی بیمار ($T_1, T_2, \dots, T_k$) تحلیل شده و هشدارهای زودهنگام بالینی جهت جلوگیری از پیشرفت عفونت و قطع عضو صادر می‌گردد.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import cv2
import numpy as np
import matplotlib.pyplot as plt

from src.config import PROJECT_ROOT
from src.preventive.trajectory_model import WoundTrajectoryModel
from src.preventive.clinical_decision_support import ClinicalDecisionSupportEngine

print("Preventive care modules loaded successfully!")

## ۱. فرمول‌بندی ریاضی شاخص التیام (Healing Index) و ریسک بیوفیلم (BDRS)
فرمول شاخص التیام:
$$HI(t) = \frac{\text{Area}(\text{Granulation}) + \text{Area}(\text{Epithelial})}{\text{Total Wound Area}}$$

فرمول نرخ کاهش مساحت زخم (Area Reduction Rate):
$$ARR(t) = \frac{A_0 - A_t}{A_0} \times 100\%$$

فرمول امتیاز ریسک زودهنگام تخریب و بیوفیلم (BDRS):

In [ ]:
# تست فرمول ریاضی روی مقادیر نمونه یک بیمار فرضی در دو ویزیت متوالی
hi_t1 = WoundTrajectoryModel.compute_healing_index(granulation_px=4500, epithelial_px=1500, total_wound_px=10000)
hi_t2 = WoundTrajectoryModel.compute_healing_index(granulation_px=3000, epithelial_px=1000, total_wound_px=12000)

risk = WoundTrajectoryModel.compute_biofilm_deterioration_score(
    slough_ratio=0.45, necrotic_ratio=0.10, delta_slough=+0.15, delta_area_pct=+20.0
)

print(f"شاخص التیام در ویزیت اول: {hi_t1 * 100:.1f}%")
print(f"شاخص التیام در ویزیت دوم (افت التیام): {hi_t2 * 100:.1f}%")
print(f"امتیاز ریسک عفونت و بیوفیلم محاسبه‌شده (BDRS): {risk} از ۱۰۰")

## ۲. نمایش ترژکتوری چندبیماری و پیش‌بینی وضعیت التیام (Longitudinal Trajectories)

In [ ]:
chart_file = PROJECT_ROOT / "results/preventive/healing_trajectory_projections.png"
if chart_file.exists():
    traj_img = cv2.cvtColor(cv2.imread(str(chart_file)), cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(16, 5), dpi=150)
    plt.imshow(traj_img)
    plt.axis('off')
    plt.title("Multi-Patient Healing Index Trajectories & Biofilm Risk Scores", fontsize=12, fontweight='bold')
    plt.show()

## ۳. موتور تصمیم‌یار بالینی (CDSS) و چارچوب بین‌المللی TIME
تولید خودکار یادداشت تصمیم‌یار برای پزشک و پیام هشدار برای بیمار در پلتفرم پزشکی از راه دور:

In [ ]:
sample_stats = {"healing_index": 0.35, "slough_ratio": 0.45, "necrotic_ratio": 0.12}
sample_forecast = {"healing_status": "STALLED_OR_EXPANDING", "estimated_days_to_closure": None}

alert = ClinicalDecisionSupportEngine.evaluate_clinical_status(
    patient_id="Patient_20",
    current_stats=sample_stats,
    trajectory_analysis=sample_forecast,
    risk_score=72.5
)

print(f"سطح خطر: {alert['color_badge']}")
print(f"فوریت مداخله: {alert['clinical_urgency']}")
print("\nدستورالعمل‌های درمانی پزشک (TIME Framework):")
for axis, advice in alert["time_framework_recommendations"].items():
    print(f"[{axis}]: {advice}")

print(f"\nپیام هشدار ارسال‌شده به گوشی بیمار:")
print(f"> {alert['patient_instructions']}")